In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Parameters
INPUT_LEN = 336
OUTPUT_LEN = 48
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-4
PATIENCE = 2

# Load data
df = pd.read_csv("uk_data_clean.csv", parse_dates=["datetime"])
df = df.sort_values(["id", "datetime"])

# Add time features
df["hour"] = df["datetime"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow"] = df["datetime"].dt.dayofweek
df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)

# Grouped by household
df_grouped = df.groupby("id")
household_ids = df["id"].unique()

# Sequence generator
def make_multivariate_sequences(features, target, input_len, output_len):
    X, y = [], []
    for i in range(len(target) - input_len - output_len + 1):
        X.append(features[i:i+input_len])
        y.append(target[i+input_len:i+input_len+output_len])
    return np.array(X), np.array(y)

# Dataset class
class MultiFeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Updated LSTM model
class DeepLSTMForecast(nn.Module):
    def __init__(self, input_size=5, hidden_size=64, num_layers=2, output_len=48, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_len)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# Init model
model = DeepLSTMForecast(input_size=5, output_len=OUTPUT_LEN).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

# Training
best_loss = float('inf')
no_improve_epochs = 0

print("Training with multivariate input...")

for epoch in range(EPOCHS):
    total_loss = 0
    count = 0
    model.train()

    for hid in tqdm(household_ids):
        try:
            df_h = df_grouped.get_group(hid).copy()
            df_h = df_h.set_index("datetime")
            df_h["target"] = df_h["target"].interpolate()

            if df_h["target"].isna().sum() > 0 or len(df_h) < 500:
                continue

            feature_cols = ["target", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
            values = df_h[feature_cols].values
            target_vals = df_h["target"].values
            n = len(values)

            train_end = int(n * 0.70)
            val_end = int(n * 0.85)

            scaler_x = MinMaxScaler()
            scaler_y = MinMaxScaler()

            train_X = scaler_x.fit_transform(values[:train_end])
            val_X = scaler_x.transform(values[train_end:val_end])
            train_y = scaler_y.fit_transform(target_vals[:train_end].reshape(-1, 1)).flatten()
            val_y = scaler_y.transform(target_vals[train_end:val_end].reshape(-1, 1)).flatten()

            X_train, y_train = make_multivariate_sequences(train_X, train_y, INPUT_LEN, OUTPUT_LEN)
            X_val, y_val = make_multivariate_sequences(val_X, val_y, INPUT_LEN, OUTPUT_LEN)

            if len(X_train) == 0 or len(X_val) == 0:
                continue

            train_loader = DataLoader(MultiFeatureDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = loss_fn(pred, yb)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * xb.size(0)
                count += xb.size(0)

        except Exception as e:
            print(f"Skipping {hid}: {e}")
            continue

    avg_loss = total_loss / count if count > 0 else float("inf")
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Training Loss: {avg_loss:.6f}")

    if avg_loss < best_loss - 1e-5:
        best_loss = avg_loss
        no_improve_epochs = 0
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved new best model.")
    else:
        no_improve_epochs += 1
        print(f"No improvement for {no_improve_epochs} epoch(s).")

    if no_improve_epochs >= PATIENCE:
        print("Early stopping triggered.")
        break

# === Evaluation ===
print("Evaluating on test set...")
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
results = []

for hid in tqdm(household_ids[:20]):
    try:
        df_h = df_grouped.get_group(hid).copy()
        df_h = df_h.set_index("datetime")
        df_h["target"] = df_h["target"].interpolate()

        if df_h["target"].isna().sum() > 0 or len(df_h) < 500:
            continue

        feature_cols = ["target", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
        values = df_h[feature_cols].values
        target_vals = df_h["target"].values
        n = len(values)

        train_end = int(n * 0.70)
        val_end = int(n * 0.85)
        test_X = values[val_end:]
        test_y = target_vals[val_end:]

        if len(test_y) < INPUT_LEN + OUTPUT_LEN:
            continue

        scaler_x = MinMaxScaler()
        scaler_y = MinMaxScaler()
        scaler_x.fit(values[:train_end])
        scaler_y.fit(target_vals[:train_end].reshape(-1, 1))

        test_X_scaled = scaler_x.transform(test_X)
        test_y_scaled = scaler_y.transform(test_y.reshape(-1, 1)).flatten()

        X_test, y_test = make_multivariate_sequences(test_X_scaled, test_y_scaled, INPUT_LEN, OUTPUT_LEN)
        if len(X_test) == 0:
            continue

        X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        with torch.no_grad():
            preds = model(X_tensor).cpu().numpy()

        preds_inv = scaler_y.inverse_transform(preds)
        y_true_inv = scaler_y.inverse_transform(y_test.reshape(-1, OUTPUT_LEN))

        rmse = np.sqrt(mean_squared_error(y_true_inv, preds_inv))
        mae = mean_absolute_error(y_true_inv, preds_inv)

        results.append({
            "id": hid,
            "rmse": rmse,
            "mae": mae,
            "forecast": preds_inv,
            "actual": y_true_inv,
        })

    except Exception as e:
        print(f"Skipping {hid} in test: {e}")
        continue

df_summary = pd.DataFrame([{"id": r["id"], "rmse": r["rmse"], "mae": r["mae"]} for r in results])
print("\nTest Summary (Top 5 by RMSE):")
print(df_summary.sort_values("rmse").head())

# Plot
if results:
    best = sorted(results, key=lambda x: x["rmse"])[0]
    plt.figure(figsize=(12, 6))
    for i in range(min(5, len(best["forecast"]))):
        plt.plot(best["forecast"][i], label=f"Forecast {i+1}", linestyle="--")
        plt.plot(best["actual"][i], label=f"Actual {i+1}")
    plt.title(f"Test Forecast - Household {best['id']} | RMSE: {best['rmse']:.4f} | MAE: {best['mae']:.4f}")
    plt.legend()
    plt.tight_layout()
    plt.show()


Using device: cuda


FileNotFoundError: [Errno 2] No such file or directory: 'uk_data_clean.csv'